# 支线 15：投机解码 — 用小模型「猜」，大模型「验证」

> **背景**：LLM 推理只能一个一个 token 串行生成，这是物理限制。但有没有办法「一次生成多个 token」？
>
> 本 Part 目标：**理解投机解码（Speculative Decoding）的核心原理——用小模型猜多个 token，大模型一次性验证。**

核心思想一句话：**让一个便宜的小模型先猜 5 个 token，大模型一次性检查这 5 个对不对。对的保留，错的扔掉重来。**

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)

### 1. 为什么不能一次生成多个 token？

回顾自回归生成：

```
Step 1: 输入 [BOS]        → 预测 token₁
Step 2: 输入 [BOS, tok₁]  → 预测 token₂
Step 3: 输入 [BOS, tok₁, tok₂] → 预测 token₃
```

每一步都依赖上一步的结果。这是**串行依赖**，物理上无法并行。

但等等——如果我能**猜**出 token₁ 是什么，不就可以提前算 token₂ 了吗？

```
正常: 算 token₁ → 等结果 → 算 token₂ → 等结果 → 算 token₃
投机: 猜 token₁=5 → 同时算 token₂(假设token₁=5) → 同时算 token₃(假设token₁=5,token₂=3)
       ↑ 如果猜对了，一次 forward 就拿到了 3 个 token！
```

**这就是投机解码的核心直觉。**

### 2. 投机解码的完整流程

需要两个模型：
- **Draft Model（草稿模型）**：小、快、便宜。比如 0.5B 参数。
- **Target Model（目标模型）**：大、慢、贵。比如 70B 参数。

流程分三步：

```
┌─────────────────────────────────────────────────┐
│  Step 1: Draft Model 猜 K 个 token              │
│                                                 │
│  输入: [BOS, 今天, 天气]                        │
│    ↓ Draft Model 自回归生成                     │
│  猜:  [真, 好, 啊, ！]  (K=4 个 token)          │
│                                                 │
├─────────────────────────────────────────────────┤
│  Step 2: Target Model 一次性验证                │
│                                                 │
│  输入: [BOS, 今天, 天气, 真, 好, 啊, ！]        │
│    ↓ Target Model 一次 forward                  │
│  输出: 每个位置的预测概率                        │
│                                                 │
│  检查:                                          │
│    位置 3: Draft 猜 "真" → Target 也认为 "真" 概率高 ✅ │
│    位置 4: Draft 猜 "好" → Target 也认为 "好" 概率高 ✅ │
│    位置 5: Draft 猜 "啊" → Target 认为 "呀" 概率更高 ❌ │
│                                                 │
├─────────────────────────────────────────────────┤
│  Step 3: 接受/拒绝                              │
│                                                 │
│  接受: "真", "好" (前 2 个猜对了)               │
│  拒绝: "啊", "！" (第 3 个猜错了，后面全丢弃)    │
│  从 Target 的分布中采样正确的 token: "呀"        │
│                                                 │
│  结果: 一次 Target forward 拿到了 3 个 token     │
│        ("真", "好", "呀")                       │
│        而不是正常情况下的 1 个 token              │
└─────────────────────────────────────────────────┘
```

### 3. 关键问题：怎么判断「接受」还是「拒绝」？

不是简单比较「Draft 猜的和 Target 最可能的是否一样」。

而是用**概率比**：

```
对 Draft 猜的每个 token x:

  p_target(x) = Target 模型认为 x 的概率
  p_draft(x)  = Draft 模型认为 x 的概率

  接受概率 = min(1, p_target(x) / p_draft(x))

  如果 p_target(x) >= p_draft(x): 一定接受 ✅
  如果 p_target(x) < p_draft(x):  以概率 p_target/p_draft 接受
```

**直觉**：
- 如果 Target 比 Draft 更确信这个 token → 接受
- 如果 Draft 过于自信但 Target 不认可 → 可能拒绝
- 这个规则保证了**最终输出的分布和只用 Target 模型完全一样**（数学可证明！）

In [ ]:
# 模拟投机解码的接受/拒绝逻辑
def speculative_accept(draft_probs, target_probs, draft_tokens):
    """
    投机解码的接受判断
    
    参数:
        draft_probs: Draft 模型对每个猜测 token 的概率 [K]
        target_probs: Target 模型对每个猜测 token 的概率 [K]
        draft_tokens: Draft 猜的 token IDs [K]
    返回:
        接受的 token 列表
    """
    accepted = []
    for i, (dp, tp, tok) in enumerate(zip(draft_probs, target_probs, draft_tokens)):
        # 接受概率
        accept_prob = min(1.0, tp / dp) if dp > 0 else 0.0
        
        # 以 accept_prob 的概率接受
        if torch.rand(1).item() < accept_prob:
            accepted.append(tok)
        else:
            # 拒绝！从这里开始后面的都丢弃
            break
    return accepted

# 模拟一次投机解码
print("=== 模拟投机解码 ===")
print()

# Draft 猜了 4 个 token，以及它自己对每个 token 的置信度
draft_tokens = [15, 23, 8, 42]
draft_probs = [0.8, 0.7, 0.6, 0.5]  # Draft 自己的概率

# Target 验证时，对这些 token 的概率
target_probs = [0.9, 0.8, 0.2, 0.1]  # Target 的概率

print(f"Draft 猜的 token: {draft_tokens}")
print(f"Draft 的概率:     {draft_probs}")
print(f"Target 的概率:    {target_probs}")
print()

for i in range(len(draft_tokens)):
    ratio = target_probs[i] / draft_probs[i]
    accept_prob = min(1.0, ratio)
    status = "✅ 一定接受" if ratio >= 1 else f"🎲 {accept_prob:.0%} 概率接受"
    print(f"  token {draft_tokens[i]}: p_target/p_draft = {ratio:.2f} → {status}")

print()
print("前两个 token Target 更确信 → 一定接受")
print("第三个 token Target 不太认可 → 大概率拒绝")

### 4. 投机解码为什么能加速？

关键数字：**Draft Model 的推理速度是 Target Model 的 10-100 倍。**

```
正常解码:
  Target 一次 forward → 1 个 token
  生成 100 token → 100 次 Target forward

投机解码:
  Draft 猜 5 个 token (5 次 Draft forward，但每次很快)
  Target 一次 forward 验证 5 个
  假设接受 3 个 → 1 次 Target forward 拿到 3 个 token
  生成 100 token → 约 33 次 Target forward
  
  加速比 ≈ 100/33 ≈ 3x
```

**加速比取决于 Draft 的准确率**：
- Draft 猜得准 → 一次接受很多 token → 加速多
- Draft 猜不准 → 频繁拒绝 → 加速少，甚至可能变慢

实际中，用一个 0.5B 的 draft 模型配合 7B 的 target 模型，通常能加速 2-3 倍。

In [ ]:
# 模拟不同接受率下的加速效果
def simulate_speedup(K=5, accept_rate=0.6, num_tokens=100):
    """
    模拟投机解码的加速比
    
    K: Draft 每次猜几个 token
    accept_rate: 每个 token 被接受的概率
    """
    target_calls = 0
    tokens_generated = 0
    
    while tokens_generated < num_tokens:
        target_calls += 1
        # 几何分布：期望接受数 = accept_rate / (1 - accept_rate)
        # 简化：直接算期望
        expected_accepted = min(K, int(1 / (1 - accept_rate)))
        tokens_generated += expected_accepted
    
    return num_tokens / target_calls

print("=== 不同接受率下的加速比 (K=5) ===")
print()
for ar in [0.3, 0.5, 0.7, 0.9]:
    speedup = simulate_speedup(K=5, accept_rate=ar)
    print(f"接受率 {ar:.0%}: 加速 {speedup:.1f}x")

print()
print("接受率越高 → 加速越多")
print("接受率 90% → 接近 5x 加速（因为一次接受约 5 个 token）")

### 5. 投机解码的变体

业界有很多变体，核心思想一样，只是「谁来当 Draft」不同：

| 变体 | Draft 模型 | 特点 |
|------|-----------|------|
| **标准投机解码** | 独立的小模型 | 需要额外训练/部署一个小模型 |
| **Self-Speculative** | 模型自己的早期层 | 不需要额外模型！用前几层的输出当 draft |
| **Medusa** | 多个额外的「头」 | 在模型上挂几个额外的预测头，同时预测未来多个 token |
| **Eagle** | 一个小的特征变换网络 | 用 target 模型的特征预测未来 token |
| **Lookahead Decoding** | 用 n-gram 匹配 | 从已生成的文本中找匹配的 n-gram 当 draft |

**Medusa 特别有意思**：在 LLM 最后一层上挂几个额外的线性头，每个头预测未来第 k 个 token。
不需要额外的 draft 模型，只需要训练几个小头。

In [ ]:
# Medusa 的核心概念示意
print("=== Medusa 头的概念 ===")
print()
print("普通 LLM 最后一层:")
print("  hidden_states → lm_head → 预测下一个 token")
print()
print("Medusa LLM 最后一层:")
print("  hidden_states → lm_head     → 预测 token_{t+1}")
print("  hidden_states → medusa_head₁ → 预测 token_{t+2}")
print("  hidden_states → medusa_head₂ → 预测 token_{t+3}")
print("  hidden_states → medusa_head₃ → 预测 token_{t+4}")
print()
print("一次 forward 同时预测未来 4 个 token！")
print("然后用 target 模型验证，和投机解码一样的接受/拒绝逻辑。")

### 6. 投机解码的局限

不是所有场景都适合投机解码：

**适合的场景**：
- 生成「套路化」内容（代码、翻译、摘要）→ Draft 猜得准 → 加速多
- 大 batch 推理（服务端）→ Target 的 forward 很贵，值得投机

**不适合的场景**：
- 创意写作 → Draft 猜不准 → 频繁拒绝 → 可能更慢
- 单条请求（本地推理）→ Draft 的开销可能超过收益
- 显存紧张 → 多一个 draft 模型占显存

**关键洞察**：投机解码不减少总计算量，只是把串行变成了「小模型串行 + 大模型并行验证」。
大模型的计算量没变，但等待时间减少了。

### 7. 投机解码 vs 普通解码：一张图总结

```
普通自回归解码:
  Target: [算]→tok₁→[算]→tok₂→[算]→tok₃→[算]→tok₄
  时间:   ████      ████      ████      ████
  4 次 Target forward，串行

投机解码:
  Draft:  [算]→g₁→[算]→g₂→[算]→g₃→[算]→g₄  (快！)
  Target:         [一次性验证 g₁g₂g₃g₄]
  时间:   ████████████████████████████████
          ↑ Draft 很快，Target 只算一次
  
  如果 g₁g₂ 接受，g₃ 拒绝:
  拿到 tok₁=g₁, tok₂=g₂, tok₃=重新采样
  1 次 Target forward 拿到 3 个 token → 3x 加速
```

---

## 投机解码支线小结

1. ✅ 自回归生成的串行依赖是物理限制，无法消除
2. ✅ 投机解码：用小模型猜多个 token，大模型一次性验证
3. ✅ 接受/拒绝用概率比：`min(1, p_target/p_draft)`，保证分布不变
4. ✅ 加速比取决于 Draft 准确率，通常 2-3x
5. ✅ 变体：Self-Speculative、Medusa、Eagle、Lookahead
6. ✅ 适合套路化内容（代码、翻译），不适合创意写作
7. ✅ 不减少总计算量，但把串行等待变成了并行验证

**一句话总结**：投机解码 = 让便宜的「实习生」先写草稿，昂贵的「老板」只检查——老板不用逐字写，效率自然高。